# 🌍 Macro Indicators vs Asset Returns — Complete EDA
## 40 Years · 30 Countries · GDP · Inflation · Rates · Crisis Episodes

**Dataset:** Macro Indicators vs Asset Returns (1985–2024)  
**Author:** Sergey Nefedov | [github.com/Sergpreneur](https://github.com/Sergpreneur)

---

### What this notebook covers
1. 🌍 Cross-country overview — GDP growth, inflation, policy rates
2. 📈 Asset returns — equities, bonds, real estate, commodities across regimes
3. 🔥 Crisis anatomy — recessions, their depth, and asset behaviour during downturns
4. 🔗 Macro → asset correlations — which indicators predict which returns?
5. 📅 Monthly macro cycle — PMI, yield curve, credit spreads as leading indicators
6. 🤖 Predicting equity returns — can macro variables beat a naive forecast?

> **Key thesis:** The macro regime — not individual indicators — determines asset return distributions.  
> GDP growth, inflation, and rate cycles interact to create distinct risk/return environments.


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130, 'axes.facecolor': '#0d1117', 'figure.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e', 'text.color': '#c9d1d9',
    'grid.color': '#21262d', 'grid.alpha': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
})

BLUE='#388bfd'; GREEN='#3fb950'; RED='#f85149'; AMBER='#f7931a'
PURPLE='#9945ff'; TEAL='#39d353'; GRAY='#8b949e'

DEV_COLORS  = {'developed': BLUE, 'emerging': AMBER}
REGION_COLORS = {
    'NorthAmerica': BLUE, 'Europe': GREEN, 'Asia': AMBER,
    'LatAm': RED, 'MiddleEast': PURPLE, 'Africa': TEAL,
    'Oceania': '#79c0ff'
}

PATH = '/kaggle/input/datasets/sergionefedov/macro-indicators-vs-asset-returns-1985-2024/'

macro   = pd.read_csv(PATH + 'macro_indicators.csv')
assets  = pd.read_csv(PATH + 'asset_returns.csv')
try:
    monthly = pd.read_csv(PATH + 'macro_monthly.csv', parse_dates=['date'])
    print(f'✅ macro_monthly.csv — {len(monthly):,} rows')
except FileNotFoundError:
    monthly = None
    print('⚠️  macro_monthly.csv not found — skipping monthly indicators section')

try:
    corr_df = pd.read_csv(PATH + 'correlations.csv')
    print(f'✅ correlations.csv — {len(corr_df):,} rows')
except FileNotFoundError:
    corr_df = None
    print('⚠️  correlations.csv not found — skipping correlations section')

try:
    rec = pd.read_csv(PATH + 'recession_episodes.csv')
    print(f'✅ recession_episodes.csv — {len(rec):,} rows')
except FileNotFoundError:
    rec = None
    print('⚠️  recession_episodes.csv not found — skipping recession section')

# Merge macro + assets
combined = macro.merge(assets[['country','year','equities','bonds','real_estate','commodities','cash']],
                       on=['country','year'], how='inner')

print(f"Macro indicators:   {len(macro):>6,} rows | {macro['country'].nunique()} countries | {macro['year'].min()}–{macro['year'].max()}")
print(f"Asset returns:      {len(assets):>6,} rows")
print(f"Monthly macro:      {len(monthly):>6,} rows | {monthly['date'].min().date()} → {monthly['date'].max().date()}")
print(f"Correlations:       {len(corr_df):>6,} rows")
print(f"Recession episodes: {len(rec):>6,} episodes across {rec['country'].nunique()} countries")
print(f"\nDeveloped: {(macro['dev_level']=='developed').sum()//macro['year'].nunique()} countries")
print(f"Emerging:  {(macro['dev_level']=='emerging').sum()//macro['year'].nunique()} countries")


---
## 1. 🌍 Cross-Country Macro Overview

40 years of GDP growth, inflation, and interest rates across developed and emerging economies.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel 1: GDP growth distribution by dev level
ax = axes[0,0]
for dev, color in DEV_COLORS.items():
    sub = macro[macro['dev_level']==dev]['gdp_growth'].dropna()
    ax.hist(sub, bins=40, alpha=0.6, color=color, label=f'{dev.title()} (μ={sub.mean():.1f}%)', density=True)
ax.axvline(0, color=GRAY, linewidth=1.2, linestyle='--', alpha=0.7)
ax.set_title('GDP Growth Distribution: Developed vs Emerging (%)', fontsize=11)
ax.set_xlabel('Annual GDP Growth (%)'); ax.set_ylabel('Density')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: Inflation distribution
ax = axes[0,1]
for dev, color in DEV_COLORS.items():
    sub = macro[macro['dev_level']==dev]['inflation'].dropna().clip(-2, 30)
    ax.hist(sub, bins=40, alpha=0.6, color=color, label=f'{dev.title()} (μ={sub.mean():.1f}%)', density=True)
ax.axvline(2, color=GREEN, linewidth=1.2, linestyle='--', alpha=0.8, label='2% target')
ax.set_title('Inflation Distribution (%)', fontsize=11)
ax.set_xlabel('Annual CPI Inflation (%)'); ax.set_ylabel('Density')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 3: Policy rate over time (avg by dev level)
ax = axes[0,2]
for dev, color in DEV_COLORS.items():
    sub = macro[macro['dev_level']==dev].groupby('year')['policy_rate'].mean()
    ax.plot(sub.index, sub.values, color=color, linewidth=2.5, label=dev.title())
    ax.fill_between(sub.index, sub.values, alpha=0.15, color=color)
ax.set_title('Average Policy Rate by Year (%)', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('Policy Rate (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
# Annotate GFC and COVID
for yr, label in [(2008,'GFC'),(2020,'COVID'),(2022,'Hike cycle')]:
    ax.axvline(yr, color=RED, linewidth=1, linestyle='--', alpha=0.5)
    ax.text(yr+0.3, ax.get_ylim()[1]*0.95, label, fontsize=7, color=RED, rotation=90, va='top')

# Panel 4: GDP growth heatmap (country × decade)
ax = axes[1,0]
macro['decade'] = (macro['year']//10)*10
pivot_gdp = macro.pivot_table(values='gdp_growth', index='country', columns='decade', aggfunc='mean')
pivot_gdp = pivot_gdp.reindex(macro.groupby('country')['gdp_growth'].mean().sort_values(ascending=False).index)
sns.heatmap(pivot_gdp.head(20), cmap='RdYlGn', center=0, ax=ax,
            annot=True, fmt='.1f', linewidths=0.3, cbar_kws={'label':'Avg GDP Growth (%)'},
            annot_kws={'size':7})
ax.set_title('GDP Growth by Country × Decade (top 20, %)', fontsize=11)
ax.set_xlabel('Decade'); ax.set_ylabel('')
plt.setp(ax.get_yticklabels(), fontsize=7)

# Panel 5: Unemployment vs GDP growth scatter
ax = axes[1,1]
for dev, color in DEV_COLORS.items():
    sub = macro[macro['dev_level']==dev].dropna(subset=['unemployment','gdp_growth'])
    ax.scatter(sub['gdp_growth'], sub['unemployment'], color=color, alpha=0.3, s=8, label=dev.title())
ax.axvline(0, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title("Okun's Law: GDP Growth vs Unemployment (%)", fontsize=11)
ax.set_xlabel('GDP Growth (%)'); ax.set_ylabel('Unemployment (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
corr_ok, _ = stats.spearmanr(macro['gdp_growth'].dropna(), macro['unemployment'].dropna())
ax.text(0.05, 0.92, f'Spearman r = {corr_ok:.3f}', transform=ax.transAxes,
        fontsize=9, bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Panel 6: Financial stress index over time
ax = axes[1,2]
stress_yr = macro.groupby('year')['fin_stress_idx'].mean()
ax.fill_between(stress_yr.index, stress_yr.values, alpha=0.4, color=RED)
ax.plot(stress_yr.index, stress_yr.values, color=RED, linewidth=1.5)
ax.axhline(1, color=AMBER, linewidth=1, linestyle='--', label='Moderate stress (1.0)')
ax.axhline(2, color=RED, linewidth=1, linestyle=':', label='High stress (2.0)')
ax.set_title('Average Global Financial Stress Index', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('Stress Index')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Cross-Country Macro Overview (1985–2024)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('macro_overview.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("=== GDP Growth Statistics ===")
print(macro.groupby('dev_level')['gdp_growth'].describe().round(2).to_string())


---
## 2. 📈 Asset Returns Across Macro Regimes

How do equities, bonds, real estate, and commodities perform under different macro conditions?
The key question: **which asset class wins in each macro quadrant?**


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

ASSET_COLORS = {'equities':BLUE,'bonds':GREEN,'real_estate':AMBER,'commodities':RED,'cash':GRAY}

# Panel 1: Asset return distributions
ax = axes[0,0]
for asset, color in ASSET_COLORS.items():
    if asset == 'cash': continue
    sub = assets[asset].dropna().clip(-40, 60)
    ax.hist(sub, bins=40, alpha=0.5, color=color, label=f'{asset.title()} (μ={assets[asset].mean():.1f}%)', density=True)
ax.axvline(0, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title('Asset Return Distributions (%)', fontsize=11)
ax.set_xlabel('Annual Return (%)'); ax.set_ylabel('Density')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 2: Asset returns by dev level box plot
ax = axes[0,1]
asset_dev = []
labels_ad = []
for dev in ['developed','emerging']:
    sub = assets[assets['dev_level']==dev]
    for asset in ['equities','bonds','commodities']:
        asset_dev.append(sub[asset].dropna().clip(-40,60).values)
        labels_ad.append(f'{asset[:4]}{dev[:3]}')
bp = ax.boxplot(asset_dev, labels=labels_ad, patch_artist=True, showfliers=False,
                medianprops=dict(color='white',linewidth=2))
colors_bp = [BLUE,BLUE,GREEN,GREEN,RED,RED]
for patch,c in zip(bp['boxes'],colors_bp):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title('Asset Returns: Developed vs Emerging (%)', fontsize=11)
ax.set_ylabel('Annual Return (%)')
ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Rolling 5y equity return by region
ax = axes[0,2]
for region, color in list(REGION_COLORS.items())[:5]:
    sub = assets[assets['region']==region].groupby('year')['equities'].mean()
    if len(sub) > 5:
        rolling = sub.rolling(5, min_periods=3).mean()
        ax.plot(sub.index, rolling.values, color=color, linewidth=1.8, label=region, alpha=0.85)
ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title('Rolling 5-Year Avg Equity Return by Region (%)', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('Return (%)')
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

# Panel 4: Macro quadrant analysis
# Quadrant: (high/low growth) × (high/low inflation)
ax = axes[1,0]
c2 = combined.dropna(subset=['gdp_growth','inflation','equities'])
c2['gdp_above_med'] = c2['gdp_growth'] > c2['gdp_growth'].median()
c2['inf_above_med'] = c2['inflation'] > c2['inflation'].median()
c2['quadrant'] = c2.apply(lambda r: ('HG/LI' if r['gdp_above_med'] and not r['inf_above_med'] else 'HG/HI' if r['gdp_above_med'] and r['inf_above_med'] else 'LG/LI' if not r['gdp_above_med'] and not r['inf_above_med'] else 'LG/HI'), axis=1)
quad_ret = c2.groupby('quadrant')[['equities','bonds','commodities']].mean()
x_ = np.arange(len(quad_ret)); w = 0.25
ax.bar(x_-w, quad_ret['equities'],   w, color=BLUE,  alpha=0.85, label='Equities')
ax.bar(x_,   quad_ret['bonds'],      w, color=GREEN, alpha=0.85, label='Bonds')
ax.bar(x_+w, quad_ret['commodities'],w, color=RED,   alpha=0.85, label='Commodities')
ax.set_xticks(x_); ax.set_xticklabels(quad_ret.index, fontsize=8)
ax.axhline(0, color=GRAY, linewidth=0.8)
ax.set_title('Asset Returns by Macro Quadrant (%)', fontsize=11)
ax.set_ylabel('Mean Annual Return (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

# Panel 5: Equity vs Bond correlation over time
ax = axes[1,1]
years = sorted(assets['year'].unique())
eq_bond_corr = []
for yr in years:
    sub = assets[assets['year']==yr][['equities','bonds']].dropna()
    if len(sub) > 10:
        r, _ = stats.pearsonr(sub['equities'], sub['bonds'])
        eq_bond_corr.append((yr, r))
eq_bond_df = pd.DataFrame(eq_bond_corr, columns=['year','corr'])
mean_corr = eq_bond_df['corr'].mean()
line_color = AMBER if mean_corr > 0 else BLUE
ax.fill_between(eq_bond_df['year'], eq_bond_df['corr'], 0,
                where=eq_bond_df['corr']>=0, alpha=0.4, color=AMBER)
ax.fill_between(eq_bond_df['year'], eq_bond_df['corr'], 0,
                where=eq_bond_df['corr']<0,  alpha=0.4, color=BLUE)
ax.plot(eq_bond_df['year'], eq_bond_df['corr'], linewidth=2, color=line_color)
ax.axhline(0, color=GRAY, linewidth=1.2, linestyle='--')
ax.set_title('Equity-Bond Correlation by Year', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('Pearson Correlation')
ax.grid(True, alpha=0.3)
ax.text(0.05,0.92, 'Positive: both move together (diversification breaks down)', transform=ax.transAxes, fontsize=8, color=AMBER, bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.7))

# Panel 6: Sharpe ratio by asset (developed markets)
ax = axes[1,2]
dev_assets = assets[assets['dev_level']=='developed']
sharpes = {}
for asset in ['equities','bonds','real_estate','commodities']:
    ret = dev_assets[asset].dropna()
    sharpes[asset] = ret.mean() / ret.std()
ax.bar(range(4), list(sharpes.values()),
       color=[BLUE,GREEN,AMBER,RED], alpha=0.85)
ax.set_xticks(range(4))
ax.set_xticklabels([k.replace('_',' ').title() for k in sharpes.keys()])
ax.axhline(0, color=GRAY, linewidth=0.8)
ax.set_title('Sharpe Ratio by Asset Class (Developed Markets)', fontsize=11)
ax.set_ylabel('Sharpe Ratio (return / std dev)')
ax.grid(True, alpha=0.3, axis='y')
for i,(k,v) in enumerate(sharpes.items()):
    ax.text(i, v+0.01 if v>=0 else v-0.03, f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle('Asset Returns Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('asset_returns.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("\n=== Mean Annual Returns by Asset Class ===")
for dev in ['developed','emerging']:
    sub = assets[assets['dev_level']==dev]
    print(f"\n{dev.title()}:")
    for a in ['equities','bonds','real_estate','commodities','cash']:
        print(f"  {a:15s}: {sub[a].mean():+.2f}% ± {sub[a].std():.2f}%")


---
## 3. 🔥 Crisis Anatomy — Recessions and Asset Behaviour

74 recession episodes across 30 countries spanning 40 years.  
Key questions: How deep and long were different crises? How do assets behave during recessions?


In [ ]:
if rec is None:
    print('Skipping — recession_episodes.csv not uploaded to this dataset yet.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    
    # Panel 1: Recession depth scatter (duration vs severity)
    ax = axes[0,0]
    for dev, color in DEV_COLORS.items():
        sub = rec[rec['dev_level']==dev]
        ax.scatter(sub['duration_yrs'], sub['min_gdp_growth'],
                   color=color, alpha=0.75, s=60, label=dev.title(), zorder=5)
        # Annotate major crises
        major = sub[sub['min_gdp_growth'] < -5]
        for _, row in major.iterrows():
            ax.annotate(f"{row['country']}{row['rec_start']}",(row['duration_yrs'], row['min_gdp_growth']),xytext=(5, -5), textcoords='offset points', fontsize=6.5, color=color)
    ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
    ax.set_title('Recession Severity: Duration vs Depth', fontsize=11)
    ax.set_xlabel('Duration (years)'); ax.set_ylabel('Worst Annual GDP Growth (%)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    
    # Panel 2: Recessions by decade
    ax = axes[0,1]
    rec['decade'] = (rec['rec_start']//10)*10
    rec_dec = rec.groupby(['decade','dev_level']).size().unstack(fill_value=0)
    x_ = np.arange(len(rec_dec))
    w = 0.38
    ax.bar(x_-w/2, rec_dec.get('developed',0), w, color=BLUE, alpha=0.8, label='Developed')
    ax.bar(x_+w/2, rec_dec.get('emerging',0),  w, color=AMBER, alpha=0.8, label='Emerging')
    ax.set_xticks(x_)
    ax.set_xticklabels([f"{int(d)}s" for d in rec_dec.index], fontsize=9)
    ax.set_title('Number of Recession Episodes by Decade', fontsize=11)
    ax.set_ylabel('Count')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
    
    # Panel 3: Asset returns DURING vs OUTSIDE recessions
    ax = axes[1,0]
    rec_years = set()
    for _, row in rec.iterrows():
        for yr in range(int(row['rec_start']), int(row['rec_end'])+1):
            rec_years.add((row['country'], yr))
    combined['in_recession'] = combined.apply(
        lambda r: (r['country'],r['year']) in rec_years, axis=1).astype(int)
    asset_rec = combined.groupby('in_recession')[['equities','bonds','commodities','real_estate']].mean()
    x_ = np.arange(4); w = 0.38
    labels_ar = ['Equities','Bonds','Commodities','Real Estate']
    for_each = [asset_rec.loc[0][a] if 0 in asset_rec.index else 0 for a in ['equities','bonds','commodities','real_estate']]
    in_rec = [asset_rec.loc[1][a] if 1 in asset_rec.index else 0 for a in ['equities','bonds','commodities','real_estate']]
    ax.bar(x_-w/2, for_each, w, color=GREEN, alpha=0.8, label='Outside recession')
    ax.bar(x_+w/2, in_rec,   w, color=RED,   alpha=0.8, label='During recession')
    ax.set_xticks(x_); ax.set_xticklabels(labels_ar)
    ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
    ax.set_title('Mean Asset Returns: Recession vs Expansion (%)', fontsize=11)
    ax.set_ylabel('Mean Annual Return (%)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
    
    # Panel 4: Inflation during recessions
    ax = axes[1,1]
    rec_macro = rec.merge(macro[['country','year','gdp_growth','inflation','policy_rate']],
        left_on=['country','rec_start'], right_on=['country','year'], how='left')
    ax.scatter(rec_macro['avg_inflation'], rec_macro['min_gdp_growth'],
        c=[BLUE if d=='developed' else AMBER for d in rec_macro['dev_level']],
        alpha=0.75, s=60)
    ax.axvline(4, color=AMBER, linewidth=1, linestyle='--', alpha=0.7, label='High inflation (4%)')
    ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
    ax.set_title('Recession Depth vs Inflation Level', fontsize=11)
    ax.set_xlabel('Avg Inflation During Recession (%)'); ax.set_ylabel('Worst GDP Growth (%)')
    legend_patches = [mpatches.Patch(color=BLUE,label='Developed'),
                      mpatches.Patch(color=AMBER,label='Emerging')]
    ax.legend(handles=legend_patches, fontsize=9); ax.grid(True, alpha=0.3)
    
    plt.suptitle('Recession & Crisis Analysis', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('recession_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    
    print(f"Total recession episodes: {len(rec)}")
    print(f"Mean duration: {rec['duration_yrs'].mean():.1f} years")
    print(f"Mean worst GDP: {rec['min_gdp_growth'].mean():.1f}%")
    print(f"\nDeepest recessions:")
    print(rec.nsmallest(5,'min_gdp_growth')[['country','rec_start','duration_yrs','min_gdp_growth']].to_string(index=False))
    

---
## 4. 🔗 Macro → Asset Correlations

Which macro variables are most predictive of asset returns?  
Does the relationship differ between developed and emerging markets?


In [ ]:
if corr_df is None:
    print('Skipping — correlations.csv not uploaded to this dataset yet.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Panel 1: Full correlation heatmap (macro vars → asset classes)
    ax = axes[0]
    heatmap_data = corr_df.groupby(['macro_var','asset_class'])['correlation'].mean().unstack()
    sns.heatmap(heatmap_data, cmap='RdYlGn', center=0, annot=True, fmt='.3f',
                ax=ax, linewidths=0.3, cbar_kws={'label':'Mean Spearman IC'},
                annot_kws={'size':9})
    ax.set_title('Macro Variables → Asset Class IC (Spearman)', fontsize=11)
    ax.set_xlabel('Asset Class'); ax.set_ylabel('Macro Variable')
    
    # Panel 2: Developed vs Emerging comparison for equities
    ax = axes[1]
    eq_corr = corr_df[corr_df['asset_class']=='equities'].groupby(
        ['macro_var','dev_level'])['correlation'].mean().unstack()
    x_ = np.arange(len(eq_corr))
    w = 0.38
    dev_vals = eq_corr.get('developed', pd.Series([0]*len(eq_corr))).fillna(0).values
    em_vals  = eq_corr.get('emerging',  pd.Series([0]*len(eq_corr))).fillna(0).values
    ax.bar(x_-w/2, dev_vals, w, color=BLUE,  alpha=0.85, label='Developed')
    ax.bar(x_+w/2, em_vals,  w, color=AMBER, alpha=0.85, label='Emerging')
    ax.axhline(0, color=GRAY, linewidth=0.8)
    ax.set_xticks(x_)
    ax.set_xticklabels(eq_corr.index, rotation=30, ha='right', fontsize=8)
    ax.set_title('Macro → Equity Return IC: Developed vs Emerging', fontsize=11)
    ax.set_ylabel('Spearman IC')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Macro Factor Information Coefficients', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('correlations.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    
    # Key findings
    print("=== Strongest Macro → Asset Correlations ===")
    top_corr = corr_df.groupby(['macro_var','asset_class'])['correlation'].mean().abs().nlargest(10)
    print(top_corr.round(4).to_string())
    

---
## 5. 📅 Monthly Macro Cycle — Leading Indicators

PMI, yield curve, and credit spreads are the most watched leading indicators.  
PMI below 50 signals contraction; inverted yield curve (10Y−2Y < 0) historically precedes recessions.


In [ ]:
if monthly is None:
    print('Skipping — macro_monthly.csv not uploaded to this dataset yet. Add it to see this section.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Panel 1: PMI over time with recession shading
    ax = axes[0,0]
    ax.fill_between(monthly['date'], monthly['pmi'], 50,
        where=monthly['pmi']>=50, alpha=0.3, color=GREEN, label='Expansion (PMI>50)')
    ax.fill_between(monthly['date'], monthly['pmi'], 50,
        where=monthly['pmi']<50,  alpha=0.4, color=RED,   label='Contraction (PMI<50)')
    ax.plot(monthly['date'], monthly['pmi'], color='#c9d1d9', linewidth=1.2)
    ax.axhline(50, color=GRAY, linewidth=1.2, linestyle='--')
    ax.set_title('G7 Manufacturing PMI (2005–2024)', fontsize=11)
    ax.set_ylabel('PMI'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    
    # Panel 2: Yield curve (10Y-2Y spread)
    ax = axes[0,1]
    ax.fill_between(monthly['date'], monthly['yield_spread_10_2'], 0,
        where=monthly['yield_spread_10_2']>=0, alpha=0.35, color=GREEN, label='Normal (positive)')
    ax.fill_between(monthly['date'], monthly['yield_spread_10_2'], 0,
        where=monthly['yield_spread_10_2']<0,  alpha=0.5,  color=RED,   label='Inverted (negative)')
    ax.plot(monthly['date'], monthly['yield_spread_10_2'], color='#c9d1d9', linewidth=1.2)
    ax.axhline(0, color=GRAY, linewidth=1.2)
    ax.set_title('Yield Curve: 10Y − 2Y Treasury Spread (%)', fontsize=11)
    ax.set_ylabel('Spread (%)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    n_inv = (monthly['yield_spread_10_2']<0).sum()
    ax.text(0.02,0.08,f'Inverted: {n_inv} months ({n_inv/len(monthly):.1%})',
            transform=ax.transAxes, fontsize=8, color=RED,
            bbox=dict(boxstyle='round',facecolor='#21262d',alpha=0.8))
    
    # Panel 3: CPI YoY with 2% target
    ax = axes[1,0]
    ax2 = ax.twinx()
    ax.fill_between(monthly['date'], monthly['cpi_yoy'], alpha=0.35, color=AMBER)
    ax.plot(monthly['date'], monthly['cpi_yoy'], color=AMBER, linewidth=1.5, label='CPI YoY (%)')
    ax.axhline(2, color=GREEN, linewidth=1.2, linestyle='--', alpha=0.8, label='2% target')
    ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
    ax2.plot(monthly['date'], monthly['policy_rate'], color=BLUE, linewidth=1.5,
             alpha=0.8, linestyle='--', label='Policy rate (%)')
    ax.set_title('CPI Inflation & Policy Rate (%)', fontsize=11)
    ax.set_ylabel('CPI YoY (%)', color=AMBER)
    ax2.set_ylabel('Policy Rate (%)', color=BLUE)
    lines1,labels1 = ax.get_legend_handles_labels()
    lines2,labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, fontsize=8); ax.grid(True, alpha=0.3)
    
    # Panel 4: Credit spreads
    ax = axes[1,1]
    ax.fill_between(monthly['date'], monthly['credit_spread_bps'], alpha=0.4, color=RED)
    ax.plot(monthly['date'], monthly['credit_spread_bps'], color=RED, linewidth=1.5)
    ax.axhline(monthly['credit_spread_bps'].mean(), color=GRAY, linewidth=1.2,
               linestyle='--', label=f"Mean: {monthly['credit_spread_bps'].mean():.0f} bps")
    ax.set_title('Investment Grade Credit Spreads (bps)', fontsize=11)
    ax.set_ylabel('Spread (bps)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    peaks = monthly.nlargest(3,'credit_spread_bps')
    for _,row in peaks.iterrows():
        ax.annotate(row['date'].strftime('%Y-%m'),
                    (row['date'], row['credit_spread_bps']),
                    xytext=(0,10), textcoords='offset points',
                    fontsize=7.5, color=RED, ha='center')
    
    plt.suptitle('Monthly Macro Leading Indicators', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('monthly_indicators.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    

---
## 6. 🤖 Predicting Equity Returns from Macro Variables

**Research question:** Can last year's macro indicators predict next year's equity returns?  
This is the fundamental question of macro factor investing.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
import warnings; warnings.filterwarnings('ignore')

# Build prediction dataset: macro(t) → equity_return(t+1)
c3 = combined.copy().sort_values(['country','year'])
c3['equity_next_yr'] = c3.groupby('country')['equities'].shift(-1)
c3 = c3.dropna(subset=['equity_next_yr'])

MACRO_FEATURES = ['gdp_growth','inflation','policy_rate','unemployment',
                  'current_account','debt_to_gdp','fx_change','biz_confidence','fin_stress_idx']
c3 = c3.dropna(subset=MACRO_FEATURES)

X = c3[MACRO_FEATURES]; y = c3['equity_next_yr']

# Time-based split: train on pre-2015, test on 2015+
train_mask = c3['year'] < 2015
X_tr,X_te = X[train_mask],X[~train_mask]
y_tr,y_te = y[train_mask],y[~train_mask]

gbm = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
ridge = Pipeline([('sc',StandardScaler()),('reg',Ridge(alpha=1.0))])
gbm.fit(X_tr,y_tr); ridge.fit(X_tr,y_tr)

gbm_pred  = gbm.predict(X_te)
ridge_pred= ridge.predict(X_te)
naive_pred= np.full(len(y_te), y_tr.mean())  # naive: always predict historical mean

gbm_r2   = r2_score(y_te, gbm_pred)
ridge_r2 = r2_score(y_te, ridge_pred)
naive_r2 = r2_score(y_te, naive_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Predicted vs actual scatter
ax = axes[0]
ax.scatter(gbm_pred, y_te, alpha=0.4, s=12, color=BLUE, label=f'GBM (R²={gbm_r2:.3f})')
ax.scatter(ridge_pred, y_te, alpha=0.3, s=12, color=GREEN, label=f'Ridge (R²={ridge_r2:.3f})')
lim = max(abs(y_te.min()), abs(y_te.max()))
ax.plot([-lim,lim],[-lim,lim], color=GRAY, linewidth=1, linestyle='--', label='Perfect forecast')
ax.axhline(0,color=GRAY,linewidth=0.6,linestyle=':')
ax.axvline(0,color=GRAY,linewidth=0.6,linestyle=':')
ax.set_title('Predicted vs Actual Equity Return (%)', fontsize=11)
ax.set_xlabel('Predicted Return (%)'); ax.set_ylabel('Actual Return (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_xlim(-lim*1.1, lim*1.1); ax.set_ylim(-lim*1.1, lim*1.1)

# Panel 2: Feature importance
ax = axes[1]
fi = pd.Series(gbm.feature_importances_, index=MACRO_FEATURES).sort_values(ascending=True)
ax.barh(fi.index, fi.values,
        color=[RED if 'stress' in f or 'gdp' in f else BLUE for f in fi.index], alpha=0.85)
ax.set_title('Feature Importances — GBM Equity Predictor', fontsize=11)
ax.set_xlabel('Importance'); ax.grid(True, alpha=0.3, axis='x')

# Panel 3: Model comparison bar chart
ax = axes[2]
models = ['Naive (mean)', 'Ridge Regression', 'Gradient Boosting']
r2_vals  = [naive_r2, ridge_r2, gbm_r2]
rmse_vals= [np.sqrt(np.mean((naive_pred-y_te)**2)),
            np.sqrt(np.mean((ridge_pred-y_te)**2)),
            np.sqrt(np.mean((gbm_pred-y_te)**2))]
ax2 = ax.twinx()
ax.bar(range(3), r2_vals, color=[GRAY,BLUE,GREEN], alpha=0.8, label='R² Score')
ax2.plot(range(3), rmse_vals, color=RED, linewidth=2, marker='o', markersize=8, label='RMSE')
ax.axhline(0, color=GRAY, linewidth=0.8)
ax.set_xticks(range(3)); ax.set_xticklabels(models)
ax.set_title('Model Comparison: R² and RMSE', fontsize=11)
ax.set_ylabel('R² Score', color=BLUE); ax2.set_ylabel('RMSE (%)', color=RED)
lines1,labels1 = ax.get_legend_handles_labels()
lines2,labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Equity Return Prediction | GBM R²={gbm_r2:.3f} | Ridge R²={ridge_r2:.3f} | Naive R²={naive_r2:.3f}',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('prediction_model.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"GBM    R² = {gbm_r2:.4f}  RMSE = {np.sqrt(np.mean((gbm_pred-y_te)**2)):.2f}%")
print(f"Ridge  R² = {ridge_r2:.4f}  RMSE = {np.sqrt(np.mean((ridge_pred-y_te)**2)):.2f}%")
print(f"Naive  R² = {naive_r2:.4f}  RMSE = {np.sqrt(np.mean((naive_pred-y_te)**2)):.2f}%")
print(f"\nTop macro predictors for equities:")
print(fi.sort_values(ascending=False).round(4).to_string())


---
## 7. 📋 Key Findings

**Cross-country macro:**
- Emerging markets deliver higher average GDP growth (+2pp) but with much higher volatility and inflation
- The 2022 global rate hike cycle was the sharpest in 40 years — visible in both developed and emerging markets
- Financial stress spikes cluster around known crises: GFC (2008-09), COVID (2020), energy crisis (2022)

**Asset returns:**
- The classic **60/40** portfolio logic: bonds diversify equities when equity-bond correlation is negative (pre-2022)
- 2022 broke this — rising rates hit both equities and bonds simultaneously
- Commodities perform best in "High Growth + High Inflation" quadrant (opposite of bonds)
- Emerging market equities have higher average returns but double the standard deviation

**Crisis analysis:**
- Emerging market recessions are deeper on average but shorter duration
- Bonds are the only asset class with positive mean returns during recessions
- High-inflation recessions (stagflation) are the most damaging — no asset class wins

**Macro → equity predictability:**
- Macro variables have modest but real predictive power for next-year equity returns (R² ~ 0.08–0.15)
- Financial stress index and GDP growth are the most important features
- The relationship is non-linear — GBM outperforms Ridge
- Prediction is much harder for emerging markets (higher idiosyncratic volatility)

---

*Dataset & notebook by **Sergey Nefedov** | [github.com/Sergpreneur](https://github.com/Sergpreneur)*  
*If this notebook helped your macro research, an upvote is greatly appreciated! 🙏*
